# KNN & PCA Assignment

Name: ____________________

Total Questions: 48 (Theoretical: 20, Practical: 28)

This notebook answers the theoretical questions and provides practical implementations for KNN and PCA.

## Theoretical Questions (1?20)

**Q1.** What is K-Nearest Neighbors (KNN) and how does it work?

**A1.** KNN is a non?parametric, instance?based algorithm. For a new point, it finds the K closest training samples (by a distance metric) and predicts the majority class (classification) or the average/weighted average target (regression).

**Q2.** What is the difference between KNN Classification and KNN Regression?

**A2.** Classification predicts a discrete label by majority vote (or weighted vote) among neighbors. Regression predicts a continuous value by averaging (or weighted averaging) the neighbors? target values.

**Q3.** What is the role of the distance metric in KNN?

**A3.** It defines ?closeness? between samples and directly determines the neighbor set. Different metrics (Euclidean, Manhattan, etc.) can change which points are considered nearest and thus the prediction.

**Q4.** What is the Curse of Dimensionality in KNN?

**A4.** As dimensions grow, distances become less informative (points look similarly far), and KNN needs much more data to find truly ?close? neighbors, often degrading performance.

**Q5.** How can we choose the best value of K in KNN?

**A5.** Use cross?validation or a validation set to evaluate multiple K values; pick the one with best generalization (e.g., highest accuracy or lowest error).

**Q6.** What are KD Tree and Ball Tree in KNN?

**A6.** They are spatial data structures that speed up nearest?neighbor search. KD Tree splits space with axis?aligned hyperplanes; Ball Tree partitions data into hyperspheres.

**Q7.** When should you use KD Tree vs. Ball Tree?

**A7.** KD Tree works well for low to moderate dimensions and axis?aligned data. Ball Tree can handle higher dimensions or non?axis?aligned distributions better.

**Q8.** What are the disadvantages of KNN?

**A8.** Slow prediction on large datasets, memory?heavy (stores all data), sensitive to irrelevant features/scale, and performance drops in high dimensions.

**Q9.** How does feature scaling affect KNN?

**A9.** KNN relies on distances; features with larger scales dominate unless scaled. Standardization or normalization is usually necessary.

**Q10.** What is PCA (Principal Component Analysis)?

**A10.** PCA is a linear dimensionality reduction technique that projects data onto orthogonal directions (principal components) that capture maximum variance.

**Q11.** How does PCA work?

**A11.** It centers data, computes the covariance matrix (or uses SVD), finds eigenvectors/eigenvalues, and projects data onto top components by largest eigenvalues.

**Q12.** What is the geometric intuition behind PCA?

**A12.** PCA finds new orthogonal axes that best fit the data cloud, maximizing variance along the first axis, then second, etc.

**Q13.** What is the difference between Feature Selection and Feature Extraction?

**A13.** Selection keeps a subset of original features; extraction creates new features (e.g., PCA components) from combinations of originals.

**Q14.** What are Eigenvalues and Eigenvectors in PCA?

**A14.** Eigenvectors are the principal component directions; eigenvalues indicate how much variance each component explains.

**Q15.** How do you decide the number of components to keep in PCA?

**A15.** Use explained variance ratio (e.g., keep 90?95%), scree plot elbow, or cross?validated performance.

**Q16.** Can PCA be used for classification?

**A16.** PCA itself is unsupervised, but its components can be used as features for classifiers (including KNN).

**Q17.** What are the limitations of PCA?

**A17.** Linear only, may lose interpretability, sensitive to scaling/outliers, and components aren?t optimized for class separation.

**Q18.** How do KNN and PCA complement each other?

**A18.** PCA reduces dimensionality/noise and can improve KNN speed and accuracy by making distances more meaningful.

**Q19.** How does KNN handle missing values in a dataset?

**A19.** KNN does not handle missing values directly; you must impute or remove missing data (e.g., KNNImputer or other strategies).

**Q20.** What are the key differences between PCA and Linear Discriminant Analysis (LDA)?

**A20.** PCA is unsupervised and maximizes variance; LDA is supervised and maximizes class separability while reducing dimensions.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris, load_wine, make_classification, make_regression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error, classification_report, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer

plt.rcParams['figure.figsize'] = (6,4)


## Practical Tasks (Q21?Q48)

In [ ]:
# Q21) Train a KNN Classifier on the Iris dataset and print model accuracy
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)
print('Iris accuracy:', accuracy_score(y_test, pred))

In [ ]:
# Q22) Train a KNN Regressor on a synthetic dataset and evaluate using MSE
X, y = make_regression(n_samples=300, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
reg = KNeighborsRegressor(n_neighbors=5)
reg.fit(X_train, y_train)
pred = reg.predict(X_test)
print('MSE:', mean_squared_error(y_test, pred))

In [ ]:
# Q23) Different distance metrics (Euclidean vs Manhattan) on Iris
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)
for metric in ['euclidean', 'manhattan']:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    print(metric, 'accuracy:', accuracy_score(y_test, pred))

In [ ]:
# Q24) Different K and visualize decision boundary (2D synthetic)
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, n_redundant=0, random_state=42)

ks = [1, 3, 7]
fig, axes = plt.subplots(1, len(ks), figsize=(15,4))

x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

for ax, k in zip(axes, ks):
    clf = KNeighborsClassifier(n_neighbors=k)
    clf.fit(X, y)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3)
    ax.scatter(X[:,0], X[:,1], c=y, s=20, edgecolor='k')
    ax.set_title(f'K={k}')
plt.tight_layout()
plt.show()

In [ ]:
# Q25) Feature scaling effect
X = iris.data
y = iris.target

X_distorted = X.copy()
X_distorted[:,0] = X_distorted[:,0] * 1000

X_train, X_test, y_train, y_test = train_test_split(X_distorted, y, test_size=0.2, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_unscaled = accuracy_score(y_test, knn.predict(X_test))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
knn.fit(X_train_s, y_train)
acc_scaled = accuracy_score(y_test, knn.predict(X_test_s))

print('Unscaled accuracy:', acc_unscaled)
print('Scaled accuracy:', acc_scaled)

In [ ]:
# Q26) PCA on synthetic data and explained variance ratio
X, _ = make_classification(n_samples=300, n_features=8, n_informative=4, random_state=42)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

pca = PCA(n_components=8)
pca.fit(X_s)
print('Explained variance ratio:', pca.explained_variance_ratio_)

In [ ]:
# Q27) PCA before KNN vs without PCA (Iris)
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_no_pca = accuracy_score(y_test, knn.predict(X_test))

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

pca = PCA(n_components=2)
X_train_p = pca.fit_transform(X_train_s)
X_test_p = pca.transform(X_test_s)

knn.fit(X_train_p, y_train)
acc_pca = accuracy_score(y_test, knn.predict(X_test_p))

print('Accuracy without PCA:', acc_no_pca)
print('Accuracy with PCA (2 comps):', acc_pca)

In [ ]:
# Q28) Hyperparameter tuning using GridSearchCV (Iris)
param_grid = {
    'n_neighbors': [3,5,7,9],
    'weights': ['uniform','distance'],
    'metric': ['euclidean','manhattan']
}

knn = KNeighborsClassifier()
cv = GridSearchCV(knn, param_grid, cv=5)
cv.fit(iris.data, iris.target)
print('Best params:', cv.best_params_)
print('Best CV score:', cv.best_score_)

In [ ]:
# Q29) Number of misclassified samples (Iris)
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)
misclassified = np.sum(pred != y_test)
print('Misclassified samples:', misclassified)

In [ ]:
# Q30) PCA cumulative explained variance (Iris)
X = StandardScaler().fit_transform(iris.data)
pca = PCA().fit(X)
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker='o')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.grid(True)
plt.show()

In [ ]:
# Q31) Weights parameter (uniform vs distance)
for w in ['uniform','distance']:
    knn = KNeighborsClassifier(n_neighbors=5, weights=w)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    print(w, 'accuracy:', accuracy_score(y_test, pred))

In [ ]:
# Q32) KNN Regressor effect of K values
X, y = make_regression(n_samples=300, n_features=5, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for k in [1,3,5,9,15]:
    reg = KNeighborsRegressor(n_neighbors=k)
    reg.fit(X_train, y_train)
    pred = reg.predict(X_test)
    print('K=', k, 'MSE=', mean_squared_error(y_test, pred))

In [ ]:
# Q33) KNN Imputation for missing values
rng = np.random.RandomState(42)
X = iris.data.copy()
missing_mask = rng.rand(*X.shape) < 0.1
X[missing_mask] = np.nan

imputer = KNNImputer(n_neighbors=3)
X_imputed = imputer.fit_transform(X)
print('Missing before:', np.isnan(X).sum(), 'Missing after:', np.isnan(X_imputed).sum())

In [ ]:
# Q34) PCA projection onto first two components (Iris)
X = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
X_p = pca.fit_transform(X)
plt.scatter(X_p[:,0], X_p[:,1], c=iris.target, cmap='viridis', edgecolor='k')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Iris projected onto first two PCs')
plt.show()

In [ ]:
# Q35) KD Tree vs Ball Tree performance (Iris)
for algo in ['kd_tree','ball_tree']:
    knn = KNeighborsClassifier(n_neighbors=5, algorithm=algo)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    print(algo, 'accuracy:', accuracy_score(y_test, pred))

In [ ]:
# Q36) Scree plot on high-dimensional data
X, _ = make_classification(n_samples=400, n_features=20, n_informative=6, random_state=42)
X = StandardScaler().fit_transform(X)
pca = PCA()
pca.fit(X)
plt.plot(np.arange(1, 21), pca.explained_variance_ratio_, marker='o')
plt.xlabel('Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Scree Plot')
plt.grid(True)
plt.show()

In [ ]:
# Q37) Precision, Recall, F1-score (Iris)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
pred = knn.predict(X_test)
print(classification_report(y_test, pred))

In [ ]:
# Q38) PCA components vs accuracy (Iris)
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

for n in [1,2,3,4]:
    pca = PCA(n_components=n)
    X_train_p = pca.fit_transform(X_train_s)
    X_test_p = pca.transform(X_test_s)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_p, y_train)
    pred = knn.predict(X_test_p)
    print('n_components=', n, 'accuracy=', accuracy_score(y_test, pred))

In [ ]:
# Q39) Different leaf_size values
for ls in [10, 20, 30, 50]:
    knn = KNeighborsClassifier(n_neighbors=5, leaf_size=ls)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    print('leaf_size', ls, 'accuracy', accuracy_score(y_test, pred))

In [ ]:
# Q40) Before vs after PCA transform (Iris)
X = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
X_p = pca.fit_transform(X)

fig, axes = plt.subplots(1,2, figsize=(10,4))
axes[0].scatter(X[:,0], X[:,1], c=iris.target, cmap='viridis', edgecolor='k')
axes[0].set_title('Original (2 features)')
axes[1].scatter(X_p[:,0], X_p[:,1], c=iris.target, cmap='viridis', edgecolor='k')
axes[1].set_title('After PCA (2 comps)')
plt.tight_layout()
plt.show()

In [ ]:
# Q41) Wine dataset KNN classification report
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.2, random_state=42, stratify=wine.target)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
pred = knn.predict(X_test_s)
print(classification_report(y_test, pred))

In [ ]:
# Q42) KNN Regressor distance metrics effect
X, y = make_regression(n_samples=300, n_features=5, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for metric in ['euclidean', 'manhattan']:
    reg = KNeighborsRegressor(n_neighbors=5, metric=metric)
    reg.fit(X_train, y_train)
    pred = reg.predict(X_test)
    print(metric, 'MSE:', mean_squared_error(y_test, pred))

In [ ]:
# Q43) ROC-AUC score (binary classification)
X, y = make_classification(n_samples=400, n_features=6, n_informative=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
proba = knn.predict_proba(X_test_s)[:,1]
roc = roc_auc_score(y_test, proba)
print('ROC-AUC:', roc)

In [ ]:
# Q44) Variance captured by each principal component (Wine)
X = StandardScaler().fit_transform(wine.data)
pca = PCA()
pca.fit(X)
plt.bar(range(1, X.shape[1]+1), pca.explained_variance_ratio_)
plt.xlabel('Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance by PC')
plt.show()

In [ ]:
# Q45) Feature selection before KNN (Iris)
X = iris.data
y = iris.target
var = X.var(axis=0)
idx = np.argsort(var)[-2:]
X_sel = X[:, idx]

X_train, X_test, y_train, y_test = train_test_split(X_sel, y, test_size=0.2, random_state=42, stratify=y)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
print('Accuracy with selected features:', accuracy_score(y_test, knn.predict(X_test)))

In [ ]:
# Q46) PCA reconstruction error after reducing dimensions
X = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
X_p = pca.fit_transform(X)
X_recon = pca.inverse_transform(X_p)
recon_error = np.mean((X - X_recon)**2)
print('Reconstruction MSE:', recon_error)

In [ ]:
# Q47) KNN decision boundary (another example)
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, n_redundant=0, random_state=0)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X, y)

x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.3)
plt.scatter(X[:,0], X[:,1], c=y, s=20, edgecolor='k')
plt.title('KNN Decision Boundary')
plt.show()

In [ ]:
# Q48) Components vs data variance (Iris)
X = StandardScaler().fit_transform(iris.data)
pca = PCA().fit(X)
var_cum = np.cumsum(pca.explained_variance_ratio_)
for i, v in enumerate(var_cum, start=1):
    print('Components:', i, 'Cumulative variance:', v)